# Bridge 6 Pilot: 100-Molecule Test

**Goal**: First signal check. Do we see any pattern between molecular symmetry (point group) and assembly complexity (MA)?

**Dataset**: 100 representative small molecules

**Outputs**:
1. Scatter plot: MA vs point group order
2. Spearman correlation + p-value
3. Summary statistics
4. Box plots by symmetry type

**Success criteria**: Signal present, correlation detected, or at least no obvious errors.

In [ ]:
import sys
sys.path.insert(0, '/c/Users/Cicada38/Projects/math_exp')

import pandas as pd
import numpy as np
from pathlib import Path

# Create directories
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('figures').mkdir(parents=True, exist_ok=True)

print("Environment ready.")

## Step 1: Generate 100-molecule dataset

In [ ]:
from src.fetch_molecules import get_qm9_sample

# Get 100 test molecules
df_molecules = get_qm9_sample(n=100, use_real_data=False)

print(f"Loaded {len(df_molecules)} molecules")
print("\nFirst 10 SMILES:")
print(df_molecules['smiles'].head(10).to_string())

## Step 2: Compute point group symmetry for each molecule

In [ ]:
from src.compute_symmetry import compute_point_group

print(f"Computing point groups for {len(df_molecules)} molecules...")

symmetry_results = []
for i, smiles in enumerate(df_molecules['smiles']):
    if i % 20 == 0:
        print(f"  {i}/{len(df_molecules)}...", end='\r')
    result = compute_point_group(smiles, tolerance=0.3)
    if result:
        result['smiles'] = smiles
        symmetry_results.append(result)

df_symmetry = pd.DataFrame(symmetry_results)
print(f"\nComputed symmetry for {len(df_symmetry)} molecules (success rate: {len(df_symmetry)/len(df_molecules)*100:.1f}%)")
print("\nPoint group distribution:")
print(df_symmetry['point_group'].value_counts())

## Step 3: Compute assembly index for each molecule

In [ ]:
from src.compute_assembly import AssemblyIndexBatcher

print(f"Computing assembly indices for {len(df_symmetry)} molecules...")

# Use heuristic for speed (real MA would require DaymudeLab assembly-theory or API)
batcher = AssemblyIndexBatcher(method='heuristic', delay_seconds=0.0)
assembly_results = batcher.compute_batch(df_symmetry['smiles'].tolist())

df_assembly = pd.DataFrame(assembly_results)
df_assembly = df_assembly[['smiles', 'assembly_index']]

print(f"\nComputed assembly indices for {len(df_assembly)} molecules")
print(f"\nAssembly Index statistics:")
print(f"  Mean: {df_assembly['assembly_index'].mean():.2f}")
print(f"  Std:  {df_assembly['assembly_index'].std():.2f}")
print(f"  Min:  {df_assembly['assembly_index'].min():.2f}")
print(f"  Max:  {df_assembly['assembly_index'].max():.2f}")

## Step 4: Merge datasets

In [ ]:
from src.merge_datasets import merge_datasets, save_dataset

df_merged = merge_datasets(df_symmetry, df_assembly)

print(f"Merged dataset: {len(df_merged)} molecules with both symmetry and assembly data")
print("\nColumns:")
print(df_merged.columns.tolist())
print("\nFirst 10 rows:")
print(df_merged.head(10))

# Save for later analysis
save_dataset(df_merged, 'data/processed/pilot_100_molecules.csv')

## Step 5: Core correlation analysis

In [ ]:
from src.analyze import test_ma_vs_symmetry, compute_summary_stats

# Test main hypothesis
result = test_ma_vs_symmetry(df_merged)

print("=" * 60)
print("MA vs Point Group Order Correlation")
print("=" * 60)
print(f"Spearman ρ:  {result['rho']:.4f}")
print(f"p-value:     {result['p_value']:.6f}")
print(f"n samples:   {result['n']}")
print(f"Status:      {result['status']}")

if result['p_value'] < 0.05:
    print("\n✓ SIGNIFICANT: Correlation detected at p < 0.05")
else:
    print("\n⚠ NO SIGNIFICANT CORRELATION: p >= 0.05")

# Summary stats
print("\n" + "=" * 60)
print("Summary Statistics")
print("=" * 60)
stats_dict = compute_summary_stats(df_merged)
for key, val in stats_dict.items():
    if key != 'point_groups':
        print(f"{key:20} : {val}")

## Step 6: Visualizations

In [ ]:
from src.analyze import plot_ma_vs_symmetry, plot_distribution, plot_symmetry_box

# Main plot: MA vs symmetry
print("Creating scatter plot: MA vs Point Group Order...")
plot_ma_vs_symmetry(df_merged, output_path='figures/01_ma_vs_symmetry.png')

# Distribution of assembly indices
print("\nCreating MA distribution histogram...")
plot_distribution(df_merged, 'assembly_index', output_path='figures/01_ma_distribution.png')

# Box plots by symmetry type
print("\nCreating box plot: MA by point group...")
plot_symmetry_box(df_merged, output_path='figures/01_ma_by_symmetry.png')

## Summary

**If correlation is significant (p < 0.05):**
- ✓ The hypothesis has merit. Proceed to full QM9 dataset (134k molecules).
- ✓ Investigate whether the relationship is monotonic or complex.

**If no correlation:**
- ⚠ Molecular symmetry and assembly complexity may be independent at this scale.
- ✓ Still publishable as a negative result (orthogonal descriptors).
- → Try controlling for molecular size, or analyze specific chemical families.

**Next steps:**
1. If signal → proceed to full QM9 (notebook 02)
2. If noise → investigate confounders (size, families, etc.)
3. Begin Thread 1 (SAI on finite groups) in parallel